In [2]:
import random
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from faker import Faker

random.seed(42)
Faker.seed(42)
fake = Faker("en_IN")

START_DATE = datetime(2026, 8, 1)
CITIES = ["Hyderabad", "Bengaluru", "Mumbai", "Delhi", "Chennai"]

# Create folders inside the repository
DATA = Path(".")

RAW_DIR = DATA / "raw"
ANSWER_DIR = DATA / "answer_key"

RAW_DIR.mkdir(parents=True, exist_ok=True)
ANSWER_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
customers = pd.DataFrame([
    {
        "CustomerID": f"CUST{i:03d}",
        "CustomerName": fake.name(),
        "Age": random.randint(20, 70),
        "City": random.choice(CITIES),
        "CustomerSince": fake.date_between(
            start_date="-8y",
            end_date="-6m"
        )
    }
    for i in range(1, 101)
])

accounts = pd.DataFrame([
    {
        "AccountID": f"ACC{i:04d}",
        "CustomerID": customer_id,
        "AccountType": random.choices(
            ["SAVINGS", "CURRENT"],
            weights=[90, 10]
        )[0],
        "OpeningBalance": round(
            random.uniform(250000, 650000), 2
        )
    }
    for i, customer_id in enumerate(
        customers["CustomerID"], start=1
    )
])

account_ids = accounts["AccountID"].tolist()

account_owner = dict(zip(
    accounts["AccountID"],
    accounts["CustomerID"]
))

balances = dict(zip(
    accounts["AccountID"],
    accounts["OpeningBalance"]
))

print("Customers:", len(customers))
print("Accounts:", len(accounts))

display(customers.head())
display(accounts.head())

Customers: 100
Accounts: 100


,CustomerID,CustomerName,Age,City,CustomerSince
0,CUST001,Aryan Maharaj,60,Hyderabad,2020-12-14
1,CUST002,Liam Chaudry,21,Mumbai,2024-08-17
2,CUST003,Pahal Balay,35,Bengaluru,2019-02-15
3,CUST004,Tejas Kaul,28,Hyderabad,2023-03-04
4,CUST005,Rushil Saini,63,Chennai,2020-09-29


,AccountID,CustomerID,AccountType,OpeningBalance
0,ACC0001,CUST001,CURRENT,316050.78
1,ACC0002,CUST002,SAVINGS,492637.57
2,ACC0003,CUST003,CURRENT,621565.40
3,ACC0004,CUST004,SAVINGS,525947.05
4,ACC0005,CUST005,SAVINGS,409596.92


In [4]:
transactions = []


def add_transfer(
    sender, receiver, amount, timestamp, transaction_type
):
    amount = round(float(amount), 2)

    if sender == receiver:
        raise ValueError("Sender and receiver must differ.")

    if sender not in account_owner or receiver not in account_owner:
        raise ValueError("Unknown account.")

    if balances[sender] < amount:
        raise ValueError(f"Insufficient balance: {sender}")

    transaction_id = f"TX{len(transactions) + 1:05d}"

    transactions.append({
        "TransactionID": transaction_id,
        "CustomerID": account_owner[sender],
        "SenderAccount": sender,
        "ReceiverAccount": receiver,
        "Amount": amount,
        "Timestamp": timestamp,
        "TransactionType": transaction_type
    })

    balances[sender] -= amount
    balances[receiver] += amount

    return transaction_id


# Generate 800 normal transactions
for _ in range(800):

    possible_senders = [
        a for a in account_ids if balances[a] > 10000
    ]

    sender = random.choice(possible_senders)

    receiver = random.choice([
        a for a in account_ids if a != sender
    ])

    timestamp = START_DATE + timedelta(
        days=random.randint(0, 89),
        hours=random.randint(7, 22),
        minutes=random.randint(0, 59)
    )

    add_transfer(
        sender,
        receiver,
        random.uniform(500, 6000),
        timestamp,
        random.choice(["UPI", "IMPS", "NEFT", "RTGS"])
    )

print("Normal transactions:", len(transactions))

Normal transactions: 800


In [5]:
answer_rows = []
used_accounts = set()


def add_answer(record_id, entity_type, scenario, case_id, label, reason):
    answer_rows.append({
        "RecordID": record_id,
        "EntityType": entity_type,
        "CaseID": case_id,
        "ScenarioID": scenario,
        "Label": label,
        "Reason": reason
    })


# S1: High-value night transfers
s1_senders = sorted(
    account_ids, key=lambda a: balances[a], reverse=True
)[:2]

for n, sender in enumerate(s1_senders, 1):
    receiver = random.choice([
        a for a in account_ids
        if a not in used_accounts and a not in s1_senders
    ])

    tx_id = add_transfer(
        sender,
        receiver,
        random.uniform(110000, 150000),
        START_DATE + timedelta(
            days=20+n, hours=random.randint(1, 4)
        ),
        "IMPS"
    )

    add_answer(
        tx_id, "TRANSACTION", "S1", f"CASE-S1-{n:02d}",
        "FRAUD", "High-value transfer during night"
    )

    used_accounts.update([sender, receiver])


# S3: Rapid transfers to multiple recipients
for n in range(1, 3):
    sender = next(
        a for a in account_ids
        if a not in used_accounts and balances[a] > 100000
    )

    receivers = random.sample([
        a for a in account_ids
        if a not in used_accounts and a != sender
    ], 3)

    start = START_DATE + timedelta(days=35+n, hours=14)
    case_id = f"CASE-S3-{n:02d}"

    for step, receiver in enumerate(receivers):
        tx_id = add_transfer(
            sender,
            receiver,
            random.uniform(2500, 6500),
            start + timedelta(minutes=step * 5),
            "UPI"
        )

        add_answer(
            tx_id, "TRANSACTION", "S3", case_id,
            "FRAUD", "Three recipients within 15 minutes"
        )

    used_accounts.update([sender] + receivers)


# S4: Mule account transfer chains
chain_pool = [
    a for a in account_ids
    if a not in used_accounts and balances[a] > 100000
]

chain_accounts = random.sample(chain_pool, 8)

for n in range(1, 3):
    chain = chain_accounts[(n-1)*4:n*4]
    start = START_DATE + timedelta(days=55+n, hours=11)
    case_id = f"CASE-S4-{n:02d}"
    amount = random.uniform(50000, 60000)

    for step in range(3):
        tx_id = add_transfer(
            chain[step],
            chain[step+1],
            amount * (0.90 ** step),
            start + timedelta(minutes=step * 10),
            "IMPS"
        )

        add_answer(
            tx_id, "TRANSACTION", "S4", case_id,
            "FRAUD", "Funds moved through a four-account chain"
        )

    used_accounts.update(chain)

print("Fraud scenarios generated.")

Fraud scenarios generated.


In [6]:
login_rows = []
normal_logins = {}

# Normal logins
for i, row in accounts.iterrows():
    account_id = row["AccountID"]
    customer_id = row["CustomerID"]
    city = customers.loc[i, "City"]

    login = {
        "LoginID": f"LG{len(login_rows)+1:04d}",
        "CustomerID": customer_id,
        "AccountID": account_id,
        "DeviceID": f"DEV{i+1:03d}",
        "IPAddress": fake.ipv4_private(),
        "Location": city,
        "Timestamp": START_DATE + timedelta(
            days=random.randint(0, 60),
            hours=random.randint(7, 22)
        ),
        "FailedAttempts": 0,
        "LoginStatus": "SUCCESS"
    }

    login_rows.append(login)
    normal_logins[account_id] = login


# S2: Account takeover
s2_accounts = random.sample([
    a for a in account_ids if a not in used_accounts
], 2)

for n, account_id in enumerate(s2_accounts, 1):
    baseline = normal_logins[account_id]
    customer_id = account_owner[account_id]

    new_city = random.choice([
        c for c in CITIES if c != baseline["Location"]
    ])

    login_id = f"LG{len(login_rows)+1:04d}"

    login_rows.append({
        "LoginID": login_id,
        "CustomerID": customer_id,
        "AccountID": account_id,
        "DeviceID": f"NEWDEV{n:03d}",
        "IPAddress": fake.ipv4_private(),
        "Location": new_city,
        "Timestamp": baseline["Timestamp"] + timedelta(days=3, hours=2),
        "FailedAttempts": 3,
        "LoginStatus": "SUCCESS"
    })

    add_answer(
        login_id, "LOGIN", "S2", f"CASE-S2-{n:02d}",
        "SUSPICIOUS", "New device and location after three failed attempts"
    )

logins = pd.DataFrame(login_rows)

print("Login events:", len(logins))

Login events: 102


In [7]:
transactions = pd.DataFrame(transactions).sort_values(
    "Timestamp"
).reset_index(drop=True)

accounts["CurrentBalance"] = accounts["AccountID"].map(
    lambda a: round(balances[a], 2)
)

answer_key = pd.DataFrame(answer_rows)

customers.to_csv(RAW_DIR / "customers.csv", index=False)
accounts.to_csv(RAW_DIR / "accounts.csv", index=False)
transactions.to_csv(RAW_DIR / "transactions.csv", index=False)
logins.to_csv(RAW_DIR / "login_activity.csv", index=False)

answer_key.to_csv(
    ANSWER_DIR / "answer_key.csv", index=False
)